# QK Attention Circuit Analysis

Detailed QK circuit analysis: eigenspectrum, positional vs content,
token preference, cross-layer composition, and pattern prediction.

## Why This Matters

QK circuit attention circuit analyzes the query-key interaction that determines attention patterns. Understanding what features drive attention — positional vs. content-based, local vs. global — reveals how the model decides which information to route where.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.qk_attention_circuit import (
    qk_eigenspectrum, qk_positional_vs_content,
    qk_token_preference, qk_composition_from_prev_layer,
    qk_pattern_prediction,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## QK Eigenspectrum

Singular value structure of the QK matrix W_Q @ W_K^T.

In [ ]:
for layer in range(4):
    print(f'Layer {layer}:')
    for head in range(4):
        result = qk_eigenspectrum(model, layer=layer, head=head)
        print(f"  Head {head}: rank={result['effective_rank']:.2f}, "
              f"spec={result['spectral_norm']:.4f}")
    print()

## Positional vs Content

Is attention driven by position or content?

In [ ]:
for layer in range(4):
    print(f'Layer {layer}:')
    for head in range(4):
        result = qk_positional_vs_content(model, tokens, layer=layer, head=head)
        pos = 'POS' if result['is_positional'] else 'content'
        print(f"  Head {head}: norm_entropy={result['mean_normalized_entropy']:.4f}, "
              f"var={result['score_variance']:.4f} [{pos}]")
    print()

## Token Preference

Which tokens does each query most prefer through the QK circuit?

In [ ]:
result = qk_token_preference(model, layer=0, head=0, token_ids=[1, 10, 30, 50, 80])
for q in result['per_query']:
    print(f"  Query tok {q['query_token']} → prefers tok {q['preferred_token']} "
          f"(score={q['preference_score']:+.4f}, self={q['self_score']:+.4f})")

## Composition from Previous Layer

How does the previous layer's OV output feed into this head's Q and K?

In [ ]:
for layer in range(1, 4):
    result = qk_composition_from_prev_layer(model, layer=layer, head=0)
    print(f"L{layer}H0 ← L{result['prev_layer']}:")
    for c in result['compositions'][:2]:
        print(f"  ← H{c['prev_head']}: Q_comp={c['q_composition']:.4f}, "
              f"K_comp={c['k_composition']:.4f}")
    print()

## Pattern Prediction

Does the QK weight matrix alone explain the actual attention pattern?

In [ ]:
for layer in range(4):
    print(f'Layer {layer}:')
    for head in range(4):
        result = qk_pattern_prediction(model, tokens, layer=layer, head=head)
        match = '✓' if result['patterns_match'] else '✗'
        print(f"  Head {head}: mean_diff={result['mean_pattern_diff']:.6f}, "
              f"max_diff={result['max_pattern_diff']:.6f} [{match}]")
    print()